# Harmonisation Evaluation Metrics

In this section, we explore several approaches for evaluating harmonisation performance in longitudinal and multi-site imaging data.

---

## Reliability-Based Metrics

We first examine:

- **Within-Subject Variability**
- **Subject Order Consistency**

These metrics are particularly useful for:

- test-retest datasets,
- travelling-subject datasets,
- or scenarios where little or no biological change is expected over time.

The goal is to evaluate the reliability and consistency of measurements across scanners, sites, or repeated acquisitions.

### Why this matters

If no true biological change is expected, large differences between repeated scans may indicate:

- scanner/site effects,
- measurement instability,
- or technical variability.

Reliable measurements are especially important in clinical and longitudinal studies where subtle biological effects are being investigated.

---

## Batch-Effect Evaluation

We then evaluate:

- **Additive Batch Effects**
- **Multiplicative Batch Effects**

These analyses can be applied to both:

- test-retest datasets,
- and true longitudinal datasets with expected biological change.

### What these tests assess

- **Additive effects** evaluate whether scanners/sites introduce systematic shifts in mean values.
- **Multiplicative effects** evaluate whether scanners/sites introduce differences in variability.

These effects are important because many harmonisation methods, including ComBat-like approaches, are specifically designed to estimate and remove additive and multiplicative batch effects while preserving biological signal.

Together, these metrics provide complementary information about:
- measurement reliability,
- preservation of biological structure,
- and residual scanner/site-related bias after harmonisation.

## 1. Within-Subject Variability

This function quantifies variability across repeated measurements within the same subject.

For subjects with **2 measurements**, variability is computed as:

$$
\mathrm{Variability}(\%) =
\frac{|x_1 - x_2|}{\mathrm{mean}(x_1, x_2)} \times 100
$$

For subjects with **more than 2 measurements**, the function computes the **coefficient of variation (CoV)**:

$$
\mathrm{CoV}(\%) =
\frac{\mathrm{SD}(x)}{\mathrm{mean}(x)} \times 100
$$

Lower values indicate better agreement across scans/sites/scanners.

In [ ]:
# Uncomment if packages are missing
# !pip install numpy pandas matplotlib
# !pip install statsmodels
# !pip install scikit-learn
# !pip install seaborn

In [ ]:
import importlib
import HarmonisationEvaluation_functions

importlib.reload(HarmonisationEvaluation_functions)

from HarmonisationEvaluation_functions import simulate_harmonisation_data
df = simulate_harmonisation_data(
    n_subjects=15,
    n_timepoints=3,
    n_features=2,
    seed=1
)
print(df)

In [ ]:
# Run code for calculating within-subject variability
from HarmonisationEvaluation_functions import WithinSubjVar_long
from HarmonisationEvaluation_plots import plot_WithinSubjVar

# df has columns: Subject, Timepoint, Site, BrainVolume
idp_matrix = df[["Region_1","Region_2"]].to_numpy()
subjects = df["Subject"].to_list()
timepoints = df["Timepoint"].to_list()

# Get within subject variability 
result = WithinSubjVar_long(
    idp_matrix=idp_matrix,
    subjects=subjects,
    timepoints=timepoints,
    idp_names=["Region_1","Region_2"]
)
print(result)

# Plot within subject variability 
plot_WithinSubjVar(
            result,
            subject_col='subject',
            limit_subjects=35,
            limit_idps_for_legend=10
            )

### Interpretation of Results

The output shows the within-subject variability (%) for each subject.

Because the simulated data represent repeated scans of the same individuals with no expected biological change, differences mainly reflect:

- scanner/site effects,
- measurement noise,
- technical variability.

Higher variability values indicate poorer agreement across repeated measurements, while lower values indicate better consistency.

In the multi-timepoint example, the metric is computed using the coefficient of variation (CoV), which summarizes variability across all repeated scans for each subject.

For harmonisation evaluation:

- High within-subject variability before harmonisation suggests strong scanner/site effects.
- Reduced variability after harmonisation indicates improved consistency across sites/scanners.

## Exercise: Extend the simulated data to multiple timepoints

Modify the simulated dataset so that each subject has **3 timepoints** instead of 2.

- Use `TP0`, `TP1`, and `TP2`
- Keep one `Site` label for each timepoint
- Add a small site-related shift so the measurements are not identical
- Run `WithinSubjVar_long()` on the new dataset

**Question:**  
What changes when the function now sees more than 2 repeated measurements per subject?

## 2. Subject Order Consistency

This function evaluates how consistently subjects retain their relative ordering across timepoints/scanners.

For each IDP, the function computes the **Spearman correlation** between measurements from two timepoints:

- high correlation → subjects maintain a similar ranking across scans
- low correlation → subject ordering changes more across scans

A permutation test is used to compare the observed correlation against correlations expected by chance.

- significant p-values indicate that the observed subject ordering is unlikely to occur randomly
- non-significant p-values suggest weaker or unstable ordering consistency

For harmonisation evaluation, strong and significant subject order consistency suggests that biological differences between subjects are preserved across scanners/timepoints.

In [ ]:
import importlib
import HarmonisationEvaluation_functions
import HarmonisationEvaluation_plots


importlib.reload(HarmonisationEvaluation_functions)
importlib.reload(HarmonisationEvaluation_plots)

from HarmonisationEvaluation_functions import SubjectOrder_long
from HarmonisationEvaluation_plots import plot_SubjectOrder


# df has columns: Subject, Timepoint, Site, BrainVolume
idp_matrix = df[["Region_1","Region_2"]].to_numpy()
subjects   = df["Subject"].to_list()
timepoints = df["Timepoint"].to_list()
features   = ["Region_1","Region_2"]

subjorder = SubjectOrder_long(idp_matrix=idp_matrix,
                                        subjects=subjects,
                                        timepoints=timepoints,
                                        idp_names=features,
                                        nPerm=100)
print(subjorder)

plot_SubjectOrder(subjorder,p_correction="bonferroni")

## Exercise: Subject Order Consistency Across Timepoints

In this exercise, we evaluate whether subjects retain a similar relative ordering across repeated scans.

### Tasks

1. Simulate a dataset with:
   - multiple subjects,
   - multiple timepoints,
   - multiple brain regions/features.

2. Run `SubjectOrder_long()` using the simulated data.

3. Visualize the results using `plot_SubjectOrder()`.

4. Inspect:
   - Spearman correlation values,
   - permutation-test p-values,
   - differences across regions and timepoint pairs.

---

### Questions

- Which timepoint pairs show the strongest consistency?
- Which regions show weaker consistency?
- What happens to the correlations if scanner/site effects become larger?
- Why is preserving subject ordering important in harmonisation?

---

### Interpretation

- High Spearman correlations indicate that subjects maintain a similar ranking across scans.
- Significant permutation-test p-values indicate that the observed consistency is stronger than expected by chance.
- Strong subject order consistency after harmonisation suggests that biological differences between subjects are preserved while scanner-related effects are reduced.

## 3. Additive Batch Effect

This test asks whether the scanner/site (`batch`) adds a systematic shift to the measurements after accounting for covariates and subject-level random effects.

A simplified model is:

$$
y_{ij} = \beta_0 + \beta^T X_{ij} + \alpha_{b(i)} + u_i + \varepsilon_{ij}
$$

where:

- $y_{ij}$ = measurement for subject $i$ at observation $j$
- $X_{ij}$ = fixed effects / covariates
- $\alpha_{b(i)}$ = additive batch/site effect
- $u_i$ = subject-specific random effect
- $\varepsilon_{ij}$ = residual error

The function compares:

$$
\text{Full model: } y \sim X + \text{batch} + (1 | \text{subject})
$$

vs.

$$
\text{Reduced model: } y \sim X + (1 | \text{subject})
$$

A significant p-value suggests that the batch/site term explains additional variation, indicating an **additive batch effect**.

For harmonisation, this is the kind of shift that methods like ComBat are designed to remove.

## 4. Multiplicative Batch Effect

This test asks whether the variance differs across batches/sites after accounting for covariates and subject-level random effects.

A simplified model is:

$$
y_{ij} = \beta_0 + \beta^T X_{ij} + \alpha_{b(i)} + u_i + \varepsilon_{ij}
$$

The key difference is that the residual spread is compared across batch groups.

After fitting the mixed model, the residuals are tested with a variance test:

$$
H_0: \sigma^2_1 = \sigma^2_2 = \cdots = \sigma^2_K
$$

vs.

$$
H_1: \text{at least one batch has different variance}
$$

The function uses the **Fligner-Killeen test** on model residuals grouped by batch.

A significant p-value suggests a **multiplicative batch effect**, meaning one site/scanner has more or less variability than another.

For harmonisation, this corresponds to a scale/variance difference rather than a mean shift.

## Interpretation

- **Additive effect significant**: batches differ in mean level.
- **Multiplicative effect significant**: batches differ in variance.
- **Both significant**: scanner/site influences both location and spread.

In a longitudinal harmonisation setting, successful correction should reduce both effects while preserving subject-level structure and true biological change.

In [ ]:
import importlib
import HarmonisationEvaluation_functions
import HarmonisationEvaluation_plots

importlib.reload(HarmonisationEvaluation_functions)
importlib.reload(HarmonisationEvaluation_plots)

from HarmonisationEvaluation_functions import simulate_longitudinal_batch_data_mixed
from HarmonisationEvaluation_plots import plot_additive_multiplicative_effects


additive_shift = {
    "Site_B": {"Region_1": 600, "Region_3": -150}
}

multiplicative_scale = {
    "Site_B": {"Region_2": 15.0},
    "Site_A": {"Region_4": 2.7}
}

df = simulate_longitudinal_batch_data_mixed(
    n_subjects=100,
    n_timepoints=3,
    n_sites=2,
    n_features=4,
    additive_shift=additive_shift,
    multiplicative_scale=multiplicative_scale,
    seed=1
)

print(df.head(10))
feature_cols = ["Region_1", "Region_2", "Region_3", "Region_4"]
plot_additive_multiplicative_effects(df, feature_cols=feature_cols, batch_col="Site")


In [ ]:
import importlib
import HarmonisationEvaluation_functions

importlib.reload(HarmonisationEvaluation_functions)

from HarmonisationEvaluation_functions import AdditiveEffect_long
from HarmonisationEvaluation_functions import MultiplicativeEffect_long

feature_names = ["Region_1","Region_2","Region_3","Region_4"]


additive_results, model_defs_add = AdditiveEffect_long(
    data=df,
    idp_names=feature_names,
    idvar="Subject",
    batchvar="Site",
    timevar="Timepoint",
    fix_eff=["Age"],
    ran_eff=["Subject"],
    do_zscore=True,
    verbose=True
)
print(additive_results)

multiplicative_results, model_defs_mul = MultiplicativeEffect_long(
    data=df,
    idp_names=feature_names,
    idvar="Subject",
    batchvar="Site",
    timevar="Timepoint",
    fix_eff=["Age"],
    ran_eff=["Subject"],
    do_zscore=True,
    verbose=True
)
print(multiplicative_results)


## Exercise: Detecting Additive and Multiplicative Batch Effects

In this exercise, you will simulate longitudinal imaging data containing scanner/site-related batch effects and evaluate them using mixed-effects models.

### Tasks

1. Simulate a dataset with:
   - multiple subjects,
   - multiple timepoints,
   - multiple imaging features,
   - subject-level covariates (e.g., Age),
   - additive batch effects for selected regions,
   - multiplicative batch effects for selected regions.

2. Visualize the simulated site effects using boxplots.

3. Run:
   - `AdditiveEffect_long()`
   - `MultiplicativeEffect_long()`

4. Compare the statistical results with the known simulated effects.

---

### Questions

- Which regions show significant additive effects?
- Which regions show significant multiplicative effects?
- Do the detected effects match the simulated batch effects?
- How do additive and multiplicative effects appear differently in the boxplots?
- Why is it important to model subject-level random effects in longitudinal data?

---

### Interpretation

- Significant additive effects suggest scanner/site-related mean shifts.
- Significant multiplicative effects suggest scanner/site-related variance differences.
- Successful harmonisation methods should reduce both types of batch effects while preserving biological variation.

## Further Reading

For additional methodological details, implementation examples, and extended discussions of harmonisation evaluation approaches, see:

- **Paper:**  
  [Harmonising Structural Brain MRI from Multiple Sites with Limited Sample Sizes](https://doi.org/10.64898/2026.04.21.26351106)

- **Tool/Repository:**  
  [More evaluation metrics](https://jake-turnbull.github.io/HarmonisationDiagnostics/)